In [ ]:
import os
import re
import math
import json
import random
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from contextlib import nullcontext
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
@dataclass
class TrainConfig:
    data_dir: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
    train_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv"
    lexicon_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv"
    bt_candidates_csv: str = "/kaggle/input/datasets/christiangoltz/akkadian-publication-reconstructed/backtranslation_candidates.csv"
    bt_synth_csv: str = ""
    reverse_model_path: str = ""
    generate_bt_sources_if_missing: bool = False
    reverse_prompt_prefix: str = "translate English to Akkadian transliteration: "
    reverse_batch_size: int = 8
    reverse_num_beams: int = 4
    reverse_max_new_tokens: int = 196

    base_model_path: str = "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1"
    output_dir: str = "/kaggle/working/byt5_bt_runs"
    summary_csv: str = "/kaggle/working/byt5_bt_training_summary.csv"
    prompt_prefix: str = "translate Akkadian to English: "
    use_domain_tags: bool = True

    seed: int = 42
    val_ratio: float = 0.12
    val_max_docs: int = 200

    max_source_length: int = 384
    max_target_length: int = 160

    bt_min_domain_score: float = 18.0
    bt_max_ratio: float = 0.75
    bt_max_rows: int = 3500
    bt_min_tokens: int = 4
    bt_max_tokens: int = 90
    bt_max_chars: int = 520

    in_domain_fraction: float = 0.45
    in_domain_min_rows: int = 250

    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    gold_epochs: float = 3.0
    bt_mixed_epochs: float = 2.0
    bt_tune_epochs: float = 2.0
    in_domain_epochs: float = 1.5
    gold_learning_rate: float = 2.5e-5
    bt_learning_rate: float = 2.0e-5
    tune_learning_rate: float = 1.5e-5
    in_domain_learning_rate: float = 1.0e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.04
    eval_num_beams: int = 4
    logging_steps: int = 25
    save_total_limit: int = 2
    early_stopping_patience: int = 2
    gradient_checkpointing: bool = True

    inference_num_beams: int = 8
    inference_num_beam_candidates: int = 4
    inference_num_sample_candidates: int = 8
    inference_top_p: float = 0.9
    inference_temperature: float = 0.7
    inference_repetition_penalty: float = 1.15

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)
        bf16_fn = getattr(torch.cuda, "is_bf16_supported", None)
        self.use_bf16 = bool(self.device.type == "cuda" and bf16_fn is not None and bf16_fn())
        self.use_fp16 = bool(self.device.type == "cuda" and not self.use_bf16)


cfg = TrainConfig()
set_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)

print("Train CSV:", cfg.train_csv)
print("BT candidates:", cfg.bt_candidates_csv)
print("BT synthetic:", cfg.bt_synth_csv or "<not set>")
print("Reverse model:", cfg.reverse_model_path or "<not set>")
print("Base model:", cfg.base_model_path)
print("Output dir:", cfg.output_dir)
print("Device:", cfg.device)
print("BF16:", cfg.use_bf16, "| FP16:", cfg.use_fp16)


In [ ]:
@dataclass
class ModelSpec:
    label: str
    model_path: str
    prompt_prefix: str = "translate Akkadian to English: "
    use_domain_tags: bool = False
    num_beams: int = 8
    num_beam_candidates: int = 4
    num_sample_candidates: int = 8
    max_new_tokens: int = 160
    top_p: float = 0.9
    temperature: float = 0.7
    repetition_penalty: float = 1.15
    weight: float = 1.0


@dataclass
class EnsembleConfig:
    data_dir: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
    train_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv"
    test_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    lexicon_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv"
    page_align_csv: str = "/kaggle/input/datasets/christiangoltz/akkadian-publication-reconstructed/reconstructed_page_level.csv"
    recon_csv: str = "/kaggle/input/datasets/christiangoltz/akkadian-publication-reconstructed/reconstructed_sentence_level.csv"
    submission_path: str = "/kaggle/working/submission.csv"
    debug_candidate_csv: str = "/kaggle/working/byt5_ensemble_candidates.csv"
    config_path: str = "/kaggle/working/byt5_ensemble_config.json"
    max_source_length: int = 384
    batch_size: int = 4
    use_translation_memory: bool = True
    use_recon_memory: bool = True
    use_page_memory: bool = True
    retrieval_topk: int = 6
    retrieval_keep: int = 3
    retrieval_min_score: float = 0.18
    retrieval_shortlist: int = 20
    retrieval_combined_min_score: float = 0.40
    retrieval_domain_weight: float = 0.30
    retrieval_token_overlap_weight: float = 0.15
    retrieval_char_ngram_min: int = 3
    retrieval_char_ngram_max: int = 5
    weak_memory_max_target_tokens: int = 80
    weak_memory_max_target_chars: int = 420
    recon_memory_min_score: float = 0.62
    page_memory_min_score: float = 0.72
    use_mbr: bool = True
    mbr_pool_cap: int = 24
    memory_candidate_weight: float = 0.35
    candidate_prior_scale: float = 0.12
    run_dev_tuning: bool = False
    tuning_seed: int = 42
    tuning_val_ratio: float = 0.15
    tuning_max_docs: int = 150
    tuning_mbr_pool_caps: List[int] = field(default_factory=lambda: [12, 16, 24, 32])
    tuned_debug_csv: str = "/kaggle/working/byt5_ensemble_tuning.csv"
    use_adaptive_beams: bool = True
    use_better_transformer: bool = True
    use_mixed_precision: bool = True
    save_candidate_debug: bool = True
    model_specs: List[ModelSpec] = field(default_factory=lambda: [
        ModelSpec(
            label="byt5-mbr-v2",
            model_path="/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
        )
    ])

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.submission_path).parent.mkdir(parents=True, exist_ok=True)
        bf16_fn = getattr(torch.cuda, "is_bf16_supported", None)
        bf16_supported = bool(bf16_fn()) if (self.device.type == "cuda" and bf16_fn is not None) else False
        self.use_bf16_amp = bool(self.use_mixed_precision and bf16_supported)
        if self.device.type != "cuda":
            self.use_better_transformer = False


config = EnsembleConfig()
DATA_DIR = config.data_dir
PAGE_ALIGN_CSV = config.page_align_csv
RECON_CSV = config.recon_csv
sub_path = os.path.join("/kaggle/working", "submission.csv")
config.submission_path = sub_path

print("Train:", config.train_csv)
print("Test:", config.test_csv)
print("Recon:", RECON_CSV)
print("Page align:", PAGE_ALIGN_CSV)
print("Submission:", config.submission_path)
print("Models:", [m.model_path for m in config.model_specs])
print("Device:", config.device)
print("BF16 autocast:", config.use_bf16_amp)


In [ ]:
SUBSCRIPT_TRANS = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
    "ₓ": "x",
})

AKK_CHAR_MAP = {
    "ā": "a2", "à": "a3",
    "é": "e2", "è": "e3",
    "í": "i2", "ì": "i3",
    "ú": "u2", "ų": "u3",
    "š": "sz", "Š": "SZ",
    "ṣ": "s,", "Ṣ": "S,",
    "ṭ": "t,", "Ṭ": "T,",
    "ḥ": "h", "Ḥ": "H",
    "ʾ": "'",
}

DET_CANON = {
    "d": "d", "mul": "mul", "ki": "ki", "lu2": "lu2", "e2": "e2", "uru": "uru", "kur": "kur",
    "mi": "mi", "m": "m", "ges": "gesz", "gesz": "gesz", "tug2": "tug2", "dub": "dub",
    "id2": "id2", "muszen": "muszen", "na4": "na4", "kusz": "kusz", "u2": "u2",
}

PROPER_NOUNS = set()


def normalize_ws(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).replace("\r", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_akk_chars(text: str) -> str:
    text = str(text)
    for src, tgt in AKK_CHAR_MAP.items():
        text = text.replace(src, tgt)
    return text.translate(SUBSCRIPT_TRANS)


def normalize_determinatives(text: str) -> str:
    def repl(match):
        inner = normalize_akk_chars(match.group(1)).lower().strip()
        inner = inner.replace("ǵ", "g").replace("ǵ", "g")
        inner = re.sub(r"\s+", "", inner)
        inner = DET_CANON.get(inner, inner)
        return "{" + inner + "}"
    return re.sub(r"\{([^{}]+)\}", repl, str(text))


def normalize_gaps(text: str) -> str:
    text = re.sub(r"\[\s*x\s*\]", " <gap> ", str(text), flags=re.I)
    text = re.sub(r"\[\s*(?:(?:\?|\.\.\.|…|x)(?:\s+(?:\?|\.\.\.|…|x))*)\s*\]", " <big_gap> ", text, flags=re.I)
    text = text.replace("…", " <big_gap> ")
    text = re.sub(r"\.\.\.", " <big_gap> ", text)
    return text


def remove_line_numbers(text: str) -> str:
    text = re.sub(r"(^|\s)\d{1,3}(?:['’]{1,2})?(?=\s)", " ", str(text))
    return text


def clean_translit(text: str) -> str:
    if pd.isna(text):
        return ""
    text = normalize_ws(text)
    text = normalize_akk_chars(text)
    text = normalize_determinatives(text)
    text = remove_line_numbers(text)
    text = normalize_gaps(text)
    text = text.replace("<big_gap>", " __BIG__ ").replace("<gap>", " __GAP__ " )
    text = text.replace("!", "").replace("?", "").replace("/", " " ).replace(":", " " )
    text = text.replace("ˇ", "").replace("˘", "")
    text = re.sub(r"\(([^)]*)\)", " ", text)
    text = text.replace("[", " " ).replace("]", " " )
    text = text.replace("<", " " ).replace(">", " " )
    text = text.replace("__BIG__", "<big_gap>").replace("__GAP__", "<gap>")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_translation(text: str) -> str:
    if pd.isna(text):
        return ""
    text = normalize_ws(text)
    text = text.replace("<", " " ).replace(">", " " )
    text = re.sub(r"\s+", " ", text).strip()
    return text


def infer_domain_tag(src: str) -> str:
    toks = str(src).split()
    gap_n = sum(1 for tok in toks if tok in {"<gap>", "<big_gap>"})
    upper_n = sum(1 for tok in toks if tok.isupper() and len(tok) > 1)
    if gap_n >= 3:
        return "<domain:gappy>"
    if upper_n >= 3:
        return "<domain:logogram>"
    if len(toks) <= 12:
        return "<domain:short>"
    if len(toks) >= 35:
        return "<domain:long>"
    return "<domain:mid>"


def plausible_pair(src: str, tgt: str) -> bool:
    src = normalize_ws(src)
    tgt = normalize_ws(tgt)
    if len(src) < 5 or len(tgt) < 5:
        return False
    if len(src.split()) < 2 or len(tgt.split()) < 2:
        return False
    if len(tgt) > 800:
        return False
    return True


def collapse_gap_runs_tokens(tokens, mode="single"):
    if mode in ("none", ""):
        return list(tokens)
    out = []
    gap_buf = []
    for tok in tokens:
        if tok in {"<gap>", "<big_gap>"}:
            gap_buf.append(tok)
            continue
        if gap_buf:
            out.append("<gap>" if mode == "single" else "<big_gap>")
            gap_buf = []
        out.append(tok)
    if gap_buf:
        out.append("<gap>" if mode == "single" else "<big_gap>")
    return out


class VectorizedPostprocessor:
    def __init__(self, aggressive=True, gap_collapse="single", empty_fallback="translation unavailable"):
        self.aggressive = bool(aggressive)
        self.gap_collapse = gap_collapse
        self.empty_fallback = empty_fallback
        self.patterns = {
            "gap_legacy": re.compile(r"(\[x\]|\(x\)|\bx\b)", re.I),
            "big_gap_legacy": re.compile(r"(\.{3,}|…|\[\.+\])"),
            "whitespace": re.compile(r"\s+"),
            "punct_space": re.compile(r"\s+([.,:;])"),
            "repeated_punct": re.compile(r"([.,:;])\1+"),
            "repeated_words": re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I),
            "quotes": re.compile(r"[\"„‟‚’]"),
            "soft_tags": re.compile(r"\((fem|plur|plural|sing|sg|pl|\?|!)\.?\)", re.I),
        }
        self.forbidden_trans = str.maketrans("", "", "()<>⟨⟩⟪⟫+")

    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations, dtype="object").fillna("").astype(str)
        s = s.apply(normalize_ws)
        s = s.apply(normalize_gaps)
        s = s.str.replace(self.patterns["gap_legacy"], "<gap>", regex=True)
        s = s.str.replace(self.patterns["big_gap_legacy"], "<gap>", regex=True)
        if self.aggressive:
            s = s.str.replace(self.patterns["soft_tags"], " ", regex=True)
            s = s.str.replace(self.patterns["quotes"], "", regex=True)
            s = s.str.translate(self.forbidden_trans)
            s = s.str.replace(self.patterns["repeated_words"], r"\1", regex=True)
            s = s.apply(lambda x: " ".join(collapse_gap_runs_tokens(x.split(), self.gap_collapse)))
            s = s.str.replace(self.patterns["punct_space"], r"\1", regex=True)
            s = s.str.replace(self.patterns["repeated_punct"], r"\1", regex=True)
        s = s.str.replace(self.patterns["whitespace"], " ", regex=True).str.strip()
        s = s.replace("", self.empty_fallback)
        return s.tolist()


postprocessor = VectorizedPostprocessor()


def load_proper_nouns(lexicon_path: str) -> set:
    if not os.path.exists(lexicon_path):
        return set()
    lex = pd.read_csv(lexicon_path)
    type_col = lex["type"].astype(str).str.upper() if "type" in lex.columns else pd.Series([""] * len(lex))
    mask = type_col.str.contains(r"PN|GN", regex=True, na=False)
    out = set()
    for col in ["form", "norm", "lexeme", "Alt_lex"]:
        if col in lex.columns:
            values = lex.loc[mask, col].dropna().astype(str)
            for value in values:
                value = normalize_akk_chars(value)
                value = re.sub(r"[-\s]+", "", value).strip().lower()
                if value:
                    out.add(value)
    return out


PROPER_NOUNS = load_proper_nouns(config.lexicon_csv)
print("Loaded proper noun lexicon entries:", len(PROPER_NOUNS))

In [ ]:
@dataclass
class ModelSpec:
    label: str
    model_path: str
    prompt_prefix: str = "translate Akkadian to English: "
    use_domain_tags: bool = False
    num_beams: int = 8
    num_beam_candidates: int = 4
    num_sample_candidates: int = 8
    max_new_tokens: int = 160
    top_p: float = 0.9
    temperature: float = 0.7
    repetition_penalty: float = 1.15
    weight: float = 1.0


@dataclass
class EnsembleConfig:
    data_dir: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
    train_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv"
    test_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    lexicon_csv: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv"
    page_align_csv: str = "/kaggle/input/datasets/christiangoltz/akkadian-publication-reconstructed/reconstructed_page_level.csv"
    recon_csv: str = "/kaggle/input/datasets/christiangoltz/akkadian-publication-reconstructed/reconstructed_sentence_level.csv"
    submission_path: str = "/kaggle/working/submission.csv"
    debug_candidate_csv: str = "/kaggle/working/byt5_ensemble_candidates.csv"
    config_path: str = "/kaggle/working/byt5_ensemble_config.json"
    max_source_length: int = 384
    batch_size: int = 4
    use_translation_memory: bool = True
    use_recon_memory: bool = True
    use_page_memory: bool = True
    retrieval_topk: int = 6
    retrieval_keep: int = 3
    retrieval_min_score: float = 0.18
    retrieval_shortlist: int = 20
    retrieval_combined_min_score: float = 0.40
    retrieval_domain_weight: float = 0.30
    retrieval_token_overlap_weight: float = 0.15
    retrieval_char_ngram_min: int = 3
    retrieval_char_ngram_max: int = 5
    weak_memory_max_target_tokens: int = 80
    weak_memory_max_target_chars: int = 420
    recon_memory_min_score: float = 0.62
    page_memory_min_score: float = 0.72
    use_mbr: bool = True
    mbr_pool_cap: int = 24
    memory_candidate_weight: float = 0.35
    candidate_prior_scale: float = 0.12
    run_dev_tuning: bool = False
    tuning_seed: int = 42
    tuning_val_ratio: float = 0.15
    tuning_max_docs: int = 150
    tuning_mbr_pool_caps: List[int] = field(default_factory=lambda: [12, 16, 24, 32])
    tuned_debug_csv: str = "/kaggle/working/byt5_ensemble_tuning.csv"
    use_adaptive_beams: bool = True
    use_better_transformer: bool = True
    use_mixed_precision: bool = True
    save_candidate_debug: bool = True
    model_specs: List[ModelSpec] = field(default_factory=lambda: [
        ModelSpec(
            label="byt5-mbr-v2",
            model_path="/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
        )
    ])

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.submission_path).parent.mkdir(parents=True, exist_ok=True)
        bf16_fn = getattr(torch.cuda, "is_bf16_supported", None)
        bf16_supported = bool(bf16_fn()) if (self.device.type == "cuda" and bf16_fn is not None) else False
        self.use_bf16_amp = bool(self.use_mixed_precision and bf16_supported)
        if self.device.type != "cuda":
            self.use_better_transformer = False


config = EnsembleConfig()
DATA_DIR = config.data_dir
PAGE_ALIGN_CSV = config.page_align_csv
RECON_CSV = config.recon_csv
sub_path = os.path.join("/kaggle/working", "submission.csv")
config.submission_path = sub_path

print("Train:", config.train_csv)
print("Test:", config.test_csv)
print("Recon:", RECON_CSV)
print("Page align:", PAGE_ALIGN_CSV)
print("Submission:", config.submission_path)
print("Models:", [m.model_path for m in config.model_specs])
print("Device:", config.device)
print("BF16 autocast:", config.use_bf16_amp)


In [ ]:
if cfg.reverse_model_path and not cfg.bt_synth_csv:
    cfg.generate_bt_sources_if_missing = True
    print("Reverse model detected without cached bt_synth_csv; enabling on-the-fly BT source generation.")
else:
    print("BT source generation mode:", "cached bt_synth_csv" if cfg.bt_synth_csv else "gold-only or cached/reverse depending on config")


In [ ]:
DOMAIN_TERMS = {
    "account", "accounts", "brother", "brothers", "caravan", "caravans", "city", "cities", "colony", "colonies",
    "contract", "contracts", "copy", "copies", "debt", "debts", "father", "house", "houses", "invoice",
    "invoices", "letter", "letters", "merchant", "merchants", "merchandise", "messenger", "mina", "minas",
    "mother", "payment", "payments", "seal", "sealed", "shekel", "shekels", "silver", "son", "sons",
    "tablet", "tablets", "textile", "textiles", "tin", "trade", "trading", "transport", "witness",
    "witnesses", "wife", "wives", "wool"
}

COMMENTARY_PATTERNS = [
    re.compile(r"\b(?:commentary|cf\.?|see|compare|chapter|appendix|bibliography|editor|edition)\b", re.I),
    re.compile(r"\b(?:to be read|better read|read not|the word is used|the use of|used frequently)\b", re.I),
    re.compile(r"\b(?:various objects|could be stored|no light on|problem of the weight|weights? being)\b", re.I),
    re.compile(r"\b(?:presumably|apparently|probably|perhaps|literally|gloss)\b", re.I),
]


def translation_domain_score(text: str) -> float:
    words = set(re.findall(r"[a-z']+", normalize_ws(text).lower()))
    score = 0.0
    score += 0.06 * len(words & DOMAIN_TERMS)
    if re.search(r"\b(?:from|to|say|send|owe|owed|pay|paid|sealed)\b", normalize_ws(text).lower()):
        score += 0.10
    return min(score, 1.5)


def commentary_penalty(text: str) -> float:
    norm = normalize_ws(text).lower()
    penalty = 0.0
    for pattern in COMMENTARY_PATTERNS:
        if pattern.search(norm):
            penalty += 0.35
    if len(norm.split()) > cfg.bt_max_tokens:
        penalty += 0.20
    if len(norm) > cfg.bt_max_chars:
        penalty += 0.12
    return penalty


def source_shape_score(text: str) -> float:
    toks = normalize_ws(text).split()
    if not toks:
        return 0.0
    gap_ratio = sum(tok in {"<gap>", "<big_gap>"} for tok in toks) / max(1, len(toks))
    score = 0.12 if 3 <= len(toks) <= 60 else 0.04
    if gap_ratio <= 0.30:
        score += 0.10
    elif gap_ratio <= 0.55:
        score += 0.04
    if re.search(r"[a-z0-9]-[a-z0-9]", text.lower()):
        score += 0.05
    return score


def row_domain_score(src: str, tgt: str) -> float:
    return float(100.0 * (translation_domain_score(tgt) + source_shape_score(src) - commentary_penalty(tgt)))


def build_forward_prompt(source: str) -> str:
    prefix = cfg.prompt_prefix
    if cfg.use_domain_tags:
        tag = infer_domain_tag(source)
        prefix = (tag + " " if tag else "") + prefix
    return prefix + source


In [ ]:
def detect_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def build_gold_dataframe() -> pd.DataFrame:
    train_df = pd.read_csv(cfg.train_csv)
    gold = train_df[["oare_id", "transliteration", "translation"]].copy()
    gold["source"] = gold["transliteration"].map(clean_translit)
    gold["target"] = gold["translation"].map(clean_translation)
    gold = gold[["oare_id", "source", "target"]]
    gold = gold[gold.apply(lambda row: plausible_pair(row["source"], row["target"]), axis=1)].copy()
    gold["domain_score"] = gold.apply(lambda row: row_domain_score(row["source"], row["target"]), axis=1)
    gold["origin"] = "gold"
    gold = gold.drop_duplicates(["source", "target"]).reset_index(drop=True)
    return gold


def split_gold_by_document(gold_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    doc_ids = [str(x) for x in gold_df["oare_id"].dropna().astype(str).unique().tolist() if str(x).strip()]
    rng = np.random.RandomState(cfg.seed)
    rng.shuffle(doc_ids)
    n_val = max(1, int(len(doc_ids) * float(cfg.val_ratio)))
    n_val = min(n_val, int(cfg.val_max_docs)) if int(cfg.val_max_docs) > 0 else n_val
    holdout_ids = set(doc_ids[:n_val])
    train_part = gold_df[~gold_df["oare_id"].astype(str).isin(holdout_ids)].copy().reset_index(drop=True)
    val_part = gold_df[gold_df["oare_id"].astype(str).isin(holdout_ids)].copy().reset_index(drop=True)
    return train_part, val_part


def load_bt_candidates() -> pd.DataFrame:
    if not cfg.bt_candidates_csv or not os.path.exists(cfg.bt_candidates_csv):
        print("BT candidates not found; BT stages will be skipped.")
        return pd.DataFrame(columns=["bt_id", "target", "domain_score", "token_count"])
    bt = pd.read_csv(cfg.bt_candidates_csv)
    target_col = detect_column(bt, ["target_en", "target", "translation", "translation_en"])
    if target_col is None:
        print("BT candidates missing target text column; BT stages will be skipped.")
        return pd.DataFrame(columns=["bt_id", "target", "domain_score", "token_count"])
    bt = bt.copy()
    bt["target"] = bt[target_col].map(clean_translation)
    if "domain_score" not in bt.columns:
        bt["domain_score"] = bt["target"].map(lambda x: 100.0 * translation_domain_score(x))
    else:
        bt["domain_score"] = pd.to_numeric(bt["domain_score"], errors="coerce").fillna(0.0)
    if "token_count" not in bt.columns:
        bt["token_count"] = bt["target"].str.split().map(len)
    bt["token_count"] = pd.to_numeric(bt["token_count"], errors="coerce").fillna(0).astype(int)
    bt = bt[bt["domain_score"] >= float(cfg.bt_min_domain_score)].copy()
    bt = bt[bt["token_count"].between(int(cfg.bt_min_tokens), int(cfg.bt_max_tokens))].copy()
    bt = bt[bt["target"].str.len() <= int(cfg.bt_max_chars)].copy()
    bt = bt[bt["target"].map(commentary_penalty) <= 0.40].copy()
    bt = bt.drop_duplicates(["target"]).reset_index(drop=True)
    bt["bt_id"] = np.arange(len(bt))
    bt["origin"] = "bt_candidate"
    return bt[["bt_id", "target", "domain_score", "token_count", "origin"]]


def load_cached_bt_sources(bt_candidates: pd.DataFrame) -> pd.DataFrame:
    if len(bt_candidates) == 0 or not cfg.bt_synth_csv or not os.path.exists(cfg.bt_synth_csv):
        return pd.DataFrame(columns=["bt_id", "source", "target", "domain_score", "origin"])
    synth = pd.read_csv(cfg.bt_synth_csv)
    source_col = detect_column(synth, ["synthetic_source", "synthetic_transliteration", "source", "transliteration"])
    target_col = detect_column(synth, ["target_en", "target", "translation", "translation_en"])
    if source_col is None or target_col is None:
        print("Cached BT synthetic CSV is missing source/target columns.")
        return pd.DataFrame(columns=["bt_id", "source", "target", "domain_score", "origin"])
    synth = synth.copy()
    synth["source"] = synth[source_col].map(clean_translit)
    synth["target"] = synth[target_col].map(clean_translation)
    if "bt_id" in synth.columns:
        merged = bt_candidates.merge(
            synth[["bt_id", "source", "target"]].copy(),
            on="bt_id",
            how="inner",
            suffixes=("", "_synth"),
        )
        merged["target"] = merged["target_synth"]
    else:
        bt_candidates = bt_candidates.copy()
        bt_candidates["target_key"] = bt_candidates["target"].map(normalize_ws)
        synth["target_key"] = synth["target"].map(normalize_ws)
        merged = bt_candidates.merge(
            synth[["target_key", "source", "target"]].copy(),
            on="target_key",
            how="inner",
            suffixes=("", "_synth"),
        )
        merged["target"] = merged["target_synth"]
    merged["origin"] = "bt_cached"
    return merged[["bt_id", "source", "target", "domain_score", "origin"]].copy()


def generate_bt_sources(bt_candidates: pd.DataFrame) -> pd.DataFrame:
    if len(bt_candidates) == 0:
        return pd.DataFrame(columns=["bt_id", "source", "target", "domain_score", "origin"])
    if not cfg.reverse_model_path or not os.path.exists(cfg.reverse_model_path):
        print("Reverse model path not set or missing; skipping on-the-fly BT generation.")
        return pd.DataFrame(columns=["bt_id", "source", "target", "domain_score", "origin"])

    print("Generating synthetic BT sources from reverse model:", cfg.reverse_model_path)
    tokenizer = AutoTokenizer.from_pretrained(cfg.reverse_model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg.reverse_model_path)
    model = model.to(cfg.device)
    model.eval()

    outputs = []
    texts = bt_candidates["target"].astype(str).tolist()
    for start in tqdm(range(0, len(texts), int(cfg.reverse_batch_size)), desc="Reverse BT"):
        batch = texts[start:start + int(cfg.reverse_batch_size)]
        prompts = [cfg.reverse_prompt_prefix + text for text in batch]
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=int(cfg.max_target_length + 64),
        )
        encoded = {k: v.to(cfg.device) for k, v in encoded.items()}
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                do_sample=False,
                num_beams=int(cfg.reverse_num_beams),
                max_new_tokens=int(cfg.reverse_max_new_tokens),
                repetition_penalty=1.05,
                use_cache=True,
            )
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))

    del model
    if cfg.device.type == "cuda":
        torch.cuda.empty_cache()

    generated_df = bt_candidates[["bt_id", "target", "domain_score"]].copy()
    generated_df["source"] = [clean_translit(text) for text in outputs]
    generated_df["origin"] = "bt_generated"
    cache_path = Path(cfg.output_dir) / "backtranslation_synthetic_sources.generated.csv"
    generated_df.to_csv(cache_path, index=False)
    print("Saved generated BT synthetic sources:", cache_path)
    return generated_df[["bt_id", "source", "target", "domain_score", "origin"]]


def build_bt_training_dataframe(bt_candidates: pd.DataFrame) -> pd.DataFrame:
    if len(bt_candidates) == 0:
        return pd.DataFrame(columns=["source", "target", "domain_score", "origin"])
    bt_df = load_cached_bt_sources(bt_candidates)
    if len(bt_df) == 0 and cfg.generate_bt_sources_if_missing:
        bt_df = generate_bt_sources(bt_candidates)
    if len(bt_df) == 0:
        print("No usable BT synthetic sources found.")
        return pd.DataFrame(columns=["source", "target", "domain_score", "origin"])
    bt_df = bt_df[bt_df.apply(lambda row: plausible_pair(row["source"], row["target"]), axis=1)].copy()
    bt_df["domain_score"] = bt_df.apply(lambda row: max(float(row.get("domain_score", 0.0)), row_domain_score(row["source"], row["target"])), axis=1)
    bt_df = bt_df.sort_values(["domain_score", "target"], ascending=[False, True])
    bt_df = bt_df.drop_duplicates(["source", "target"]).reset_index(drop=True)
    return bt_df[["source", "target", "domain_score", "origin"]]


def select_in_domain_subset(gold_train_df: pd.DataFrame) -> pd.DataFrame:
    if len(gold_train_df) == 0:
        return gold_train_df.copy()
    scored = gold_train_df.copy()
    scored["in_domain_score"] = scored["domain_score"].astype(float)
    scored = scored.sort_values(["in_domain_score", "target"], ascending=[False, True]).reset_index(drop=True)
    keep_n = max(int(len(scored) * float(cfg.in_domain_fraction)), int(cfg.in_domain_min_rows))
    keep_n = min(keep_n, len(scored))
    subset = scored.iloc[:keep_n].copy().reset_index(drop=True)
    subset["origin"] = subset["origin"].astype(str) + "+in_domain"
    return subset


gold_df = build_gold_dataframe()
gold_train_df, gold_val_df = split_gold_by_document(gold_df)
bt_candidates_df = load_bt_candidates()
bt_df = build_bt_training_dataframe(bt_candidates_df)

bt_cap = min(
    len(bt_df),
    int(len(gold_train_df) * float(cfg.bt_max_ratio)) if len(gold_train_df) else 0,
    int(cfg.bt_max_rows),
)
bt_df = bt_df.head(bt_cap).copy().reset_index(drop=True)
in_domain_train_df = select_in_domain_subset(gold_train_df)

print("Gold total:", len(gold_df))
print("Gold train:", len(gold_train_df))
print("Gold val:", len(gold_val_df))
print("BT candidates:", len(bt_candidates_df))
print("BT usable rows:", len(bt_df))
print("In-domain gold rows:", len(in_domain_train_df))
if len(bt_df):
    print(bt_df[["domain_score", "origin"]].head())


In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_source_length: int, max_target_length: int):
        self.sources = frame["source"].astype(str).tolist()
        self.targets = frame["target"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.model_inputs = tokenizer(
            self.sources,
            max_length=max_source_length,
            truncation=True,
        )
        self.label_inputs = tokenizer(
            self.targets,
            max_length=max_target_length,
            truncation=True,
        )

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        item = {k: self.model_inputs[k][idx] for k in self.model_inputs}
        item["labels"] = self.label_inputs["input_ids"][idx]
        return item


def corpus_bleu(predictions: List[str], references: List[str], max_n: int = 4) -> float:
    def get_ngrams(tokens: List[str], n: int) -> Dict[tuple, int]:
        counts: Dict[tuple, int] = {}
        if len(tokens) < n:
            return counts
        for i in range(len(tokens) - n + 1):
            ng = tuple(tokens[i:i + n])
            counts[ng] = counts.get(ng, 0) + 1
        return counts

    total_hyp_len = 0
    total_ref_len = 0
    clipped = [0.0] * max_n
    totals = [0.0] * max_n
    for hyp, ref in zip(predictions, references):
        hyp_tok = normalize_ws(hyp).split()
        ref_tok = normalize_ws(ref).split()
        total_hyp_len += len(hyp_tok)
        total_ref_len += len(ref_tok)
        for n in range(1, max_n + 1):
            hyp_ng = get_ngrams(hyp_tok, n)
            ref_ng = get_ngrams(ref_tok, n)
            for ng, cnt in hyp_ng.items():
                clipped[n - 1] += min(cnt, ref_ng.get(ng, 0))
                totals[n - 1] += cnt
    if total_hyp_len == 0:
        return 0.0
    bp = min(1.0, math.exp(1.0 - (total_ref_len / max(total_hyp_len, 1)))) if total_ref_len > 0 else 1.0
    log_avg = 0.0
    for n in range(max_n):
        if n == 0:
            if totals[n] == 0:
                return 0.0
            precision = clipped[n] / totals[n]
            if precision == 0:
                return 0.0
        else:
            precision = (clipped[n] + 1.0) / (totals[n] + 1.0)
        log_avg += math.log(max(precision, 1e-100)) / max_n
    return 100.0 * bp * math.exp(log_avg)


def corpus_chrfpp(predictions: List[str], references: List[str], max_char_n: int = 6, max_word_n: int = 2, beta: float = 2.0) -> float:
    def char_ngrams(text: str, n: int) -> Dict[str, int]:
        counts: Dict[str, int] = {}
        for i in range(len(text) - n + 1):
            ng = text[i:i + n]
            counts[ng] = counts.get(ng, 0) + 1
        return counts

    def get_ngrams(tokens: List[str], n: int) -> Dict[tuple, int]:
        counts: Dict[tuple, int] = {}
        if len(tokens) < n:
            return counts
        for i in range(len(tokens) - n + 1):
            ng = tuple(tokens[i:i + n])
            counts[ng] = counts.get(ng, 0) + 1
        return counts

    total_prec_num = 0.0
    total_prec_den = 0.0
    total_rec_num = 0.0
    total_rec_den = 0.0
    for hyp, ref in zip(predictions, references):
        hyp = normalize_ws(hyp)
        ref = normalize_ws(ref)
        for n in range(1, max_char_n + 1):
            hyp_ng = char_ngrams(hyp, n)
            ref_ng = char_ngrams(ref, n)
            matched = sum(min(hyp_ng.get(ng, 0), ref_ng.get(ng, 0)) for ng in hyp_ng)
            total_prec_num += matched
            total_prec_den += sum(hyp_ng.values())
            total_rec_num += matched
            total_rec_den += sum(ref_ng.values())
        hyp_tok = hyp.split()
        ref_tok = ref.split()
        for n in range(1, max_word_n + 1):
            hyp_ng = get_ngrams(hyp_tok, n)
            ref_ng = get_ngrams(ref_tok, n)
            matched = sum(min(hyp_ng.get(ng, 0), ref_ng.get(ng, 0)) for ng in hyp_ng)
            total_prec_num += matched
            total_prec_den += sum(hyp_ng.values())
            total_rec_num += matched
            total_rec_den += sum(ref_ng.values())
    precision = total_prec_num / max(total_prec_den, 1e-100)
    recall = total_rec_num / max(total_rec_den, 1e-100)
    if precision + recall == 0:
        return 0.0
    beta2 = beta * beta
    return 100.0 * (1 + beta2) * precision * recall / (beta2 * precision + recall)


def metric_fn_builder(tokenizer):
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        pred_text = [normalize_ws(x) for x in tokenizer.batch_decode(predictions, skip_special_tokens=True)]
        label_text = [normalize_ws(x) for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
        bleu = corpus_bleu(pred_text, label_text)
        chrfpp = corpus_chrfpp(pred_text, label_text)
        geo = math.sqrt(max(bleu, 0.0) * max(chrfpp, 0.0))
        return {
            "bleu": bleu,
            "chrfpp": chrfpp,
            "geo_mean_bleu_chrfpp": geo,
        }
    return compute_metrics


def combine_training_frames(frames: List[pd.DataFrame]) -> pd.DataFrame:
    frames = [frame.copy() for frame in frames if frame is not None and len(frame)]
    if not frames:
        return pd.DataFrame(columns=["source", "target", "origin"])
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.drop_duplicates(["source", "target"]).reset_index(drop=True)
    return combined


def save_inference_config(model_dir: Path, stage_name: str, metrics: Dict[str, float], mix_info: Dict[str, int]):
    payload = {
        "stage_name": stage_name,
        "prompt_prefix": cfg.prompt_prefix,
        "use_domain_tags": cfg.use_domain_tags,
        "num_beams": cfg.inference_num_beams,
        "num_beam_candidates": cfg.inference_num_beam_candidates,
        "num_sample_candidates": cfg.inference_num_sample_candidates,
        "max_new_tokens": cfg.max_target_length,
        "top_p": cfg.inference_top_p,
        "temperature": cfg.inference_temperature,
        "repetition_penalty": cfg.inference_repetition_penalty,
        "validation_metrics": {k: float(v) for k, v in metrics.items() if isinstance(v, (int, float, np.floating))},
        "training_mix": {k: int(v) for k, v in mix_info.items()},
    }
    with open(model_dir / "inference_config.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def train_stage(stage_name: str, init_model_path: str, train_df: pd.DataFrame, eval_df: pd.DataFrame, learning_rate: float, epochs: float, mix_info: Dict[str, int]) -> Dict[str, object]:
    stage_dir = Path(cfg.output_dir) / stage_name
    stage_dir.mkdir(parents=True, exist_ok=True)
    print(f"\\n===== Stage: {stage_name} =====")
    print("Init model:", init_model_path)
    print("Train rows:", len(train_df), "| Eval rows:", len(eval_df))

    tokenizer = AutoTokenizer.from_pretrained(init_model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(init_model_path)
    if cfg.gradient_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False

    train_inputs = train_df.copy()
    eval_inputs = eval_df.copy()
    train_inputs["source"] = train_inputs["source"].map(build_forward_prompt)
    eval_inputs["source"] = eval_inputs["source"].map(build_forward_prompt)

    train_ds = TranslationDataset(train_inputs, tokenizer, cfg.max_source_length, cfg.max_target_length)
    eval_ds = TranslationDataset(eval_inputs, tokenizer, cfg.max_source_length, cfg.max_target_length)

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(stage_dir),
        do_train=True,
        do_eval=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=int(cfg.logging_steps),
        per_device_train_batch_size=int(cfg.per_device_train_batch_size),
        per_device_eval_batch_size=int(cfg.per_device_eval_batch_size),
        gradient_accumulation_steps=int(cfg.gradient_accumulation_steps),
        learning_rate=float(learning_rate),
        # num_train_epochs=float(epochs),
        num_train_epochs=7,
        predict_with_generate=True,
        generation_num_beams=int(cfg.eval_num_beams),
        generation_max_length=int(cfg.max_target_length),
        load_best_model_at_end=True,
        metric_for_best_model="geo_mean_bleu_chrfpp",
        greater_is_better=True,
        save_total_limit=int(cfg.save_total_limit),
        warmup_ratio=float(cfg.warmup_ratio),
        weight_decay=float(cfg.weight_decay),
        bf16=bool(cfg.use_bf16),
        fp16=bool(cfg.use_fp16),
        gradient_checkpointing=bool(cfg.gradient_checkpointing),
        report_to=[],
        remove_unused_columns=False,
        optim="adafactor",
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
        compute_metrics=metric_fn_builder(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=int(cfg.early_stopping_patience))],
    )

    trainer.train()
    metrics = trainer.evaluate(metric_key_prefix="eval")

    best_dir = stage_dir / "best_model"
    best_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    save_inference_config(best_dir, stage_name, metrics, mix_info)

    result = {
        "stage_name": stage_name,
        "model_dir": str(best_dir),
        "train_rows": int(len(train_df)),
        "eval_rows": int(len(eval_df)),
        "learning_rate": float(learning_rate),
        "epochs": float(epochs),
        "mix_info": json.dumps(mix_info, ensure_ascii=False, sort_keys=True),
    }
    for key, value in metrics.items():
        if isinstance(value, (int, float, np.floating)):
            result[key] = float(value)
    return result


In [ ]:
results = []

gold_result = train_stage(
    stage_name="gold_only",
    init_model_path=cfg.base_model_path,
    train_df=gold_train_df,
    eval_df=gold_val_df,
    learning_rate=cfg.gold_learning_rate,
    epochs=cfg.gold_epochs,
    mix_info={"gold": len(gold_train_df)},
)
results.append(gold_result)

if len(bt_df):
    bt_mixed_train_df = combine_training_frames([gold_train_df, bt_df])
    bt_mixed_result = train_stage(
        stage_name="bt_mixed_pretrain",
        init_model_path=cfg.base_model_path,
        train_df=bt_mixed_train_df,
        eval_df=gold_val_df,
        learning_rate=cfg.bt_learning_rate,
        epochs=cfg.bt_mixed_epochs,
        mix_info={"gold": len(gold_train_df), "bt": len(bt_df)},
    )

    bt_tuned_result = train_stage(
        stage_name="bt_tuned",
        init_model_path=bt_mixed_result["model_dir"],
        train_df=gold_train_df,
        eval_df=gold_val_df,
        learning_rate=cfg.tune_learning_rate,
        epochs=cfg.bt_tune_epochs,
        mix_info={"gold": len(gold_train_df), "bt": len(bt_df)},
    )
    results.append(bt_tuned_result)

    if len(in_domain_train_df):
        bt_in_domain_result = train_stage(
            stage_name="bt_tuned_in_domain",
            init_model_path=bt_tuned_result["model_dir"],
            train_df=in_domain_train_df,
            eval_df=gold_val_df,
            learning_rate=cfg.in_domain_learning_rate,
            epochs=cfg.in_domain_epochs,
            mix_info={"gold_in_domain": len(in_domain_train_df), "bt": len(bt_df)},
        )
        results.append(bt_in_domain_result)
else:
    print("BT stages skipped because no usable synthetic BT rows were available.")

summary_df = pd.DataFrame(results)
summary_df.to_csv(cfg.summary_csv, index=False)
print("Saved training summary:", cfg.summary_csv)

ensemble_payload = {
    "models": []
}
for row in results:
    if row["stage_name"] not in {"gold_only", "bt_tuned", "bt_tuned_in_domain"}:
        continue
    ensemble_payload["models"].append({
        "label": row["stage_name"],
        "model_path": row["model_dir"],
        "prompt_prefix": cfg.prompt_prefix,
        "use_domain_tags": cfg.use_domain_tags,
        "weight": 1.0,
        "num_beams": cfg.inference_num_beams,
        "num_beam_candidates": cfg.inference_num_beam_candidates,
        "num_sample_candidates": cfg.inference_num_sample_candidates,
        "max_new_tokens": cfg.max_target_length,
        "top_p": cfg.inference_top_p,
        "temperature": cfg.inference_temperature,
        "repetition_penalty": cfg.inference_repetition_penalty,
    })

model_specs_path = Path(cfg.output_dir) / "model_specs.template.json"
with open(model_specs_path, "w", encoding="utf-8") as f:
    json.dump(ensemble_payload, f, indent=2, ensure_ascii=False)

print("Saved ensemble template:", model_specs_path)
summary_df


In [ ]:
generated_bt_cache = Path(cfg.output_dir) / "backtranslation_synthetic_sources.generated.csv"
if generated_bt_cache.exists():
    print("Generated BT synthetic sources cache:", generated_bt_cache)

trained_model_specs = []
if "results" in globals():
    for row in results:
        if row.get("stage_name") not in {"gold_only", "bt_tuned", "bt_tuned_in_domain"}:
            continue
        trained_model_specs.append(
            ModelSpec(
                label=str(row["stage_name"]),
                model_path=str(row["model_dir"]),
                prompt_prefix=cfg.prompt_prefix,
                use_domain_tags=bool(cfg.use_domain_tags),
                num_beams=int(cfg.inference_num_beams),
                num_beam_candidates=int(cfg.inference_num_beam_candidates),
                num_sample_candidates=int(cfg.inference_num_sample_candidates),
                max_new_tokens=int(cfg.max_target_length),
                top_p=float(cfg.inference_top_p),
                temperature=float(cfg.inference_temperature),
                repetition_penalty=float(cfg.inference_repetition_penalty),
                weight=1.0,
            )
        )

# Optional: add extra attached checkpoints here.
extra_model_specs = [
    # ModelSpec(
    #     label="extra-forward-model",
    #     model_path="/kaggle/input/my-extra-forward-model/best_model",
    #     prompt_prefix="translate Akkadian to English: ",
    #     use_domain_tags=True,
    #     weight=1.0,
    # ),
]

selected_specs = trained_model_specs + extra_model_specs
if selected_specs:
    config.model_specs = selected_specs

config.submission_path = "/kaggle/working/submission.csv"
config.debug_candidate_csv = "/kaggle/working/byt5_ensable_bt_full_candidates.csv"
config.config_path = "/kaggle/working/byt5_ensable_bt_full_config.json"

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Ensemble models selected:", [spec.label for spec in config.model_specs])
for spec in config.model_specs:
    print(" -", spec.label, "->", spec.model_path)


In [ ]:
train_df = pd.read_csv(config.train_csv)
test_df = pd.read_csv(config.test_csv)

DOMAIN_TERMS = {
    "account", "accounts", "brother", "brothers", "caravan", "caravans", "city", "cities", "colony", "colonies",
    "contract", "contracts", "copy", "copies", "debt", "debts", "father", "house", "houses", "invoice",
    "invoices", "letter", "letters", "merchant", "merchants", "merchandise", "messenger", "mina", "minas",
    "mother", "payment", "payments", "seal", "sealed", "shekel", "shekels", "silver", "son", "sons",
    "tablet", "tablets", "textile", "textiles", "tin", "trade", "trading", "transport", "witness",
    "witnesses", "wife", "wives", "wool"
}
DOMAIN_PRIORITY_TERMS = {
    "caravan", "caravans", "contract", "contracts", "debt", "debts", "invoice", "invoices", "letter",
    "letters", "merchant", "merchants", "merchandise", "mina", "minas", "payment", "payments", "seal",
    "sealed", "shekel", "shekels", "silver", "tablet", "tablets", "textile", "textiles", "tin", "trade",
    "trading", "witness", "witnesses", "wool"
}
COMMENTARY_PATTERNS = [
    (re.compile(r"\b(?:see|cf\.?|compare|chapter|appendix|bibliography|edition|editor|commentary)\b", re.I), 0.40),
    (re.compile(r"\b(?:to be read|better read|read not|the word is used|the use of|used frequently|various objects|could be stored|no light on|problem of the weight|weights? being)\b", re.I), 0.60),
    (re.compile(r"\b(?:presumably|apparently|probably|perhaps|literally|gloss)\b", re.I), 0.18),
    (re.compile(r"\b(?:text no\.?|this text|in text|the word|denoted by|proper-?ty of)\b", re.I), 0.32),
]


def memory_word_set(text: str) -> set:
    return set(re.findall(r"[a-z']+", normalize_ws(text).lower()))


def source_token_keyset(text: str) -> set:
    out = set()
    for tok in normalize_ws(text).split():
        tok = tok.lower().strip()
        if tok in {"<gap>", "<big_gap>"}:
            continue
        tok = tok.strip("{}")
        tok = re.sub(r"[^a-z0-9,]+", "", tok)
        if tok:
            out.add(tok)
    return out


def translation_domain_score(text: str) -> float:
    toks = memory_word_set(text)
    hits = len(toks & DOMAIN_TERMS)
    priority_hits = len(toks & DOMAIN_PRIORITY_TERMS)
    score = min(0.44, 0.05 * hits + 0.08 * priority_hits)
    norm = normalize_ws(text).lower()
    if re.search(r"\b(?:from|to|thus|say|copy|send|owe|owed|pay|paid)\b", norm):
        score += 0.04
    return score


def commentary_penalty(text: str) -> float:
    norm = normalize_ws(text).lower()
    penalty = 0.0
    for pattern, value in COMMENTARY_PATTERNS:
        if pattern.search(norm):
            penalty += value
    if len(norm.split()) > int(config.weak_memory_max_target_tokens):
        penalty += 0.18
    if len(norm) > int(config.weak_memory_max_target_chars):
        penalty += 0.12
    if norm.count("(") >= 2:
        penalty += 0.08
    return penalty


def source_memory_quality(src: str) -> float:
    toks = normalize_ws(src).split()
    if not toks:
        return 0.0
    gap_ratio = sum(tok in {"<gap>", "<big_gap>"} for tok in toks) / max(1, len(toks))
    upper_n = sum(tok.isupper() and len(tok) > 1 for tok in toks)
    det_n = src.count("{")
    score = 0.18
    if 3 <= len(toks) <= 60:
        score += 0.12
    elif len(toks) <= 90:
        score += 0.06
    if gap_ratio <= 0.30:
        score += 0.10
    elif gap_ratio <= 0.55:
        score += 0.04
    else:
        score -= 0.10
    if re.search(r"[a-z0-9]-[a-z0-9]", src.lower()):
        score += 0.06
    if upper_n + det_n > 0:
        score += 0.04
    return max(0.0, score)


def pair_balance_score(src: str, tgt: str) -> float:
    src_n = max(1, len(normalize_ws(src).split()))
    tgt_n = max(1, len(normalize_ws(tgt).split()))
    ratio = tgt_n / src_n
    if 0.35 <= ratio <= 3.80:
        return 0.16
    if 0.20 <= ratio <= 5.50:
        return 0.06
    return -0.12


def metadata_memory_score(origin: str, row: pd.Series) -> float:
    score = 0.0
    match_conf = normalize_ws(row.get("match_confidence", "")).lower()
    if match_conf == "high":
        score += 0.14
    elif match_conf == "medium":
        score += 0.06
    elif match_conf:
        score -= 0.04

    source_lang = normalize_ws(row.get("source_language", "")).lower()
    if not source_lang or source_lang in {"en", "eng", "english"}:
        score += 0.12
    else:
        score -= 0.22

    status = normalize_ws(row.get("translation_status", "")).lower()
    if status in {"translated", "machine_translated", "llm_translated", "translated_to_en"}:
        score += 0.10
    elif status in {"pass_through", "english", "en"}:
        score += 0.04
    elif status:
        score -= 0.04

    if origin == "page":
        score -= 0.04
    return score


def score_weak_memory_row(source: str, target: str, origin: str, row: pd.Series) -> float:
    score = 0.0
    score += source_memory_quality(source)
    score += pair_balance_score(source, target)
    score += translation_domain_score(target)
    score += metadata_memory_score(origin, row)
    score -= commentary_penalty(target)
    return round(max(0.0, min(1.5, score)), 4)


def filter_weak_memory_frame(frame: pd.DataFrame, origin: str, min_score: float) -> pd.DataFrame:
    cols = ["oare_id", "source", "target", "origin", "memory_score"]
    if frame is None or len(frame) == 0:
        return pd.DataFrame(columns=cols)
    frame = frame.copy()
    frame["oare_id"] = frame["oare_id"].astype(str) if "oare_id" in frame.columns else ""
    if "source_language" in frame.columns:
        lang = frame["source_language"].fillna("").astype(str).str.lower()
    else:
        lang = pd.Series([""] * len(frame), index=frame.index)
    if "translation_status" in frame.columns:
        status = frame["translation_status"].fillna("").astype(str).str.lower()
    else:
        status = pd.Series([""] * len(frame), index=frame.index)
    english_ok = lang.isin(["", "en", "eng", "english"])
    translated_ok = status.str.contains("trans", na=False)
    frame = frame[english_ok | translated_ok].copy()
    if len(frame) == 0:
        return pd.DataFrame(columns=cols)
    frame = frame[frame.apply(lambda row: plausible_pair(row["source"], row["target"]), axis=1)].copy()
    if len(frame) == 0:
        return pd.DataFrame(columns=cols)
    frame = frame[frame["target"].str.len() <= int(config.weak_memory_max_target_chars)].copy()
    frame = frame[frame["target"].str.split().map(len).between(2, int(config.weak_memory_max_target_tokens))].copy()
    frame["origin"] = origin
    frame["memory_score"] = frame.apply(lambda row: score_weak_memory_row(row["source"], row["target"], origin, row), axis=1)
    frame = frame[frame["memory_score"] >= float(min_score)].copy()
    if len(frame) == 0:
        return pd.DataFrame(columns=cols)
    frame = frame.sort_values(["memory_score", "target"], ascending=[False, True])
    frame = frame.drop_duplicates(["source", "target"], keep="first").reset_index(drop=True)
    return frame[cols]


gold_memory = train_df[["oare_id", "transliteration", "translation"]].copy()
gold_memory["source"] = gold_memory["transliteration"].map(clean_translit)
gold_memory["target"] = gold_memory["translation"].map(clean_translation)
gold_memory = gold_memory[["oare_id", "source", "target"]]
gold_memory = gold_memory[gold_memory.apply(lambda row: plausible_pair(row["source"], row["target"]), axis=1)].copy()
gold_memory["origin"] = "gold"
gold_memory["memory_score"] = 2.0

memory_frames = [gold_memory]
recon_memory = pd.DataFrame(columns=gold_memory.columns)
page_memory = pd.DataFrame(columns=gold_memory.columns)

if config.use_recon_memory and os.path.exists(RECON_CSV):
    recon_df = pd.read_csv(RECON_CSV)
    src_col = next((c for c in ["translit_sentence", "source", "transliteration"] if c in recon_df.columns), None)
    tgt_col = next((c for c in ["translation_sentence_en", "target", "translation_en", "translation"] if c in recon_df.columns), None)
    if src_col and tgt_col:
        recon_memory = recon_df.copy()
        recon_memory["source"] = recon_memory[src_col].map(clean_translit)
        recon_memory["target"] = recon_memory[tgt_col].map(clean_translation)
        recon_memory = filter_weak_memory_frame(recon_memory, origin="recon", min_score=float(config.recon_memory_min_score))
        if len(recon_memory):
            memory_frames.append(recon_memory)

if config.use_page_memory and os.path.exists(PAGE_ALIGN_CSV):
    page_df = pd.read_csv(PAGE_ALIGN_CSV)
    if "oare_id" in page_df.columns and "translation_en" in page_df.columns and "oare_id" in train_df.columns:
        page_join = train_df[["oare_id", "transliteration"]].dropna().drop_duplicates("oare_id")
        page_memory = page_df.merge(page_join, on="oare_id", how="inner")
        page_memory["source"] = page_memory["transliteration"].map(clean_translit)
        page_memory["target"] = page_memory["translation_en"].map(clean_translation)
        page_memory = filter_weak_memory_frame(page_memory, origin="page", min_score=float(config.page_memory_min_score))
        if len(page_memory):
            memory_frames.append(page_memory)

memory_df = pd.concat(memory_frames, ignore_index=True)
memory_df = memory_df.sort_values(["source", "target", "memory_score"], ascending=[True, True, False])
memory_df = memory_df.drop_duplicates(["source", "target"], keep="first").reset_index(drop=True)

test_df = test_df.copy()
test_df["source"] = test_df["transliteration"].map(clean_translit)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Gold memory:", len(gold_memory))
print("Recon memory kept:", len(recon_memory))
print("Page memory kept:", len(page_memory))
print("Combined memory:", len(memory_df))
if len(memory_df):
    print(memory_df["origin"].value_counts(dropna=False).to_dict())
    print(memory_df.groupby("origin")["memory_score"].agg(["count", "mean", "min", "max"]).round(3))
print(test_df[["id", "text_id", "source"]].head())


In [ ]:
def dedup_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        key = normalize_ws(item)
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(key)
    return out


def get_ngrams(tokens: List[str], n: int) -> Dict[tuple, int]:
    counts: Dict[tuple, int] = {}
    if len(tokens) < n:
        return counts
    for i in range(len(tokens) - n + 1):
        ng = tuple(tokens[i:i + n])
        counts[ng] = counts.get(ng, 0) + 1
    return counts


def sentence_bleu(hypothesis: str, reference: str, max_n: int = 4) -> float:
    hyp_tok = normalize_ws(hypothesis).split()
    ref_tok = normalize_ws(reference).split()
    if not hyp_tok or not ref_tok:
        return 0.0
    bp = min(1.0, math.exp(1.0 - len(ref_tok) / max(len(hyp_tok), 1)))
    log_avg = 0.0
    for n in range(1, max_n + 1):
        hyp_ng = get_ngrams(hyp_tok, n)
        ref_ng = get_ngrams(ref_tok, n)
        clipped = 0
        total = 0
        for ng, cnt in hyp_ng.items():
            clipped += min(cnt, ref_ng.get(ng, 0))
            total += cnt
        if n == 1:
            if total == 0:
                return 0.0
            precision = clipped / total
            if precision == 0:
                return 0.0
        else:
            precision = (clipped + 1) / (total + 1)
        log_avg += math.log(max(precision, 1e-100)) / max_n
    return bp * math.exp(log_avg)


def char_ngrams(text: str, n: int) -> Dict[str, int]:
    counts: Dict[str, int] = {}
    for i in range(len(text) - n + 1):
        ng = text[i:i + n]
        counts[ng] = counts.get(ng, 0) + 1
    return counts


def sentence_chrfpp(hypothesis: str, reference: str, max_char_n: int = 6, max_word_n: int = 2, beta: float = 2.0) -> float:
    hyp = normalize_ws(hypothesis)
    ref = normalize_ws(reference)
    if not hyp or not ref:
        return 0.0
    total_prec_num = 0.0
    total_prec_den = 0.0
    total_rec_num = 0.0
    total_rec_den = 0.0
    for n in range(1, max_char_n + 1):
        hyp_ng = char_ngrams(hyp, n)
        ref_ng = char_ngrams(ref, n)
        matched = sum(min(hyp_ng.get(ng, 0), ref_ng.get(ng, 0)) for ng in hyp_ng)
        total_prec_num += matched
        total_prec_den += sum(hyp_ng.values())
        total_rec_num += matched
        total_rec_den += sum(ref_ng.values())
    hyp_tok = hyp.split()
    ref_tok = ref.split()
    for n in range(1, max_word_n + 1):
        hyp_ng = get_ngrams(hyp_tok, n)
        ref_ng = get_ngrams(ref_tok, n)
        matched = sum(min(hyp_ng.get(ng, 0), ref_ng.get(ng, 0)) for ng in hyp_ng)
        total_prec_num += matched
        total_prec_den += sum(hyp_ng.values())
        total_rec_num += matched
        total_rec_den += sum(ref_ng.values())
    precision = total_prec_num / max(total_prec_den, 1e-100)
    recall = total_rec_num / max(total_rec_den, 1e-100)
    if precision + recall == 0:
        return 0.0
    beta2 = beta * beta
    return 100.0 * (1 + beta2) * precision * recall / (beta2 * precision + recall)


def competition_metric(hypothesis: str, reference: str) -> float:
    bleu = sentence_bleu(hypothesis, reference)
    chrfpp = sentence_chrfpp(hypothesis, reference) / 100.0
    if bleu <= 0 or chrfpp <= 0:
        return 0.0
    return math.sqrt(bleu * chrfpp)


def source_token_overlap(query_tokens: set, candidate_tokens: set) -> float:
    if not query_tokens or not candidate_tokens:
        return 0.0
    return len(query_tokens & candidate_tokens) / max(1, min(len(query_tokens), len(candidate_tokens)))


def build_memory_bundle(memory_df_local: pd.DataFrame) -> Dict[str, object]:
    memory_df_local = memory_df_local.copy()
    if "memory_score" not in memory_df_local.columns:
        memory_df_local["memory_score"] = 1.0
    memory_df_local["memory_score"] = memory_df_local["memory_score"].fillna(0.0).astype(float)
    memory_df_local = memory_df_local.sort_values(["source", "memory_score", "target"], ascending=[True, False, True])
    memory_df_local = memory_df_local.drop_duplicates(["source", "target"], keep="first").reset_index(drop=True)
    exact_memory_local: Dict[str, List[str]] = {}
    if len(memory_df_local):
        for src, group in memory_df_local.groupby("source", sort=False):
            ranked = group.sort_values(["memory_score", "target"], ascending=[False, True])
            exact_memory_local[src] = ranked["target"].astype(str).tolist()
    source_token_sets_local = [source_token_keyset(src) for src in memory_df_local["source"].astype(str).tolist()]
    score_array_local = memory_df_local["memory_score"].astype(float).to_numpy() if len(memory_df_local) else np.zeros(0, dtype=float)
    score_max_local = float(score_array_local.max()) if len(score_array_local) else 1.0
    vectorizer_local = None
    memory_matrix_local = None
    if config.use_translation_memory and len(memory_df_local) > 0:
        vectorizer_local = TfidfVectorizer(
            analyzer="char",
            ngram_range=(config.retrieval_char_ngram_min, config.retrieval_char_ngram_max),
            min_df=1,
        )
        memory_matrix_local = vectorizer_local.fit_transform(memory_df_local["source"].astype(str).tolist())
    return {
        "memory_df": memory_df_local,
        "exact_memory": exact_memory_local,
        "vectorizer": vectorizer_local,
        "memory_matrix": memory_matrix_local,
        "source_token_sets": source_token_sets_local,
        "score_array": score_array_local,
        "score_max": max(score_max_local, 1e-6),
    }


def sort_candidates_by_prior(cands: List[str], prior_scores: Dict[str, float] = None) -> List[str]:
    cands = dedup_keep_order(cands)
    prior_scores = {} if prior_scores is None else prior_scores
    order = {cand: idx for idx, cand in enumerate(cands)}
    return sorted(cands, key=lambda cand: (-float(prior_scores.get(cand, 0.0)), order[cand]))


def mbr_pick(cands: List[str], prior_scores: Dict[str, float] = None, pool_cap: int = None, prior_scale: float = None) -> str:
    prior_scores = {} if prior_scores is None else prior_scores
    cands = sort_candidates_by_prior(cands, prior_scores)
    if pool_cap is None:
        pool_cap = config.mbr_pool_cap
    if prior_scale is None:
        prior_scale = config.candidate_prior_scale
    if pool_cap is not None:
        cands = cands[: int(pool_cap)]
    if not cands:
        return "translation unavailable"
    if len(cands) == 1:
        return cands[0]
    scores = [[0.0] * len(cands) for _ in range(len(cands))]
    for i in range(len(cands)):
        for j in range(i + 1, len(cands)):
            score = competition_metric(cands[i], cands[j])
            scores[i][j] = score
            scores[j][i] = score
    max_prior = max(float(prior_scores.get(cand, 0.0)) for cand in cands) if cands else 0.0
    best_idx = 0
    best_avg = -1.0
    for i in range(len(cands)):
        avg = sum(scores[i]) / max(1, len(cands) - 1)
        if max_prior > 0 and float(prior_scale) > 0:
            avg += float(prior_scale) * (float(prior_scores.get(cands[i], 0.0)) / max_prior)
        if avg > best_avg:
            best_avg = avg
            best_idx = i
    return cands[best_idx]


def autocast_context():
    if config.use_bf16_amp and config.device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def adaptive_beam_size(clean_sources: List[str], base_beams: int) -> int:
    if not config.use_adaptive_beams:
        return int(base_beams)
    max_len = max((len(str(src).split()) for src in clean_sources), default=0)
    if max_len <= 12:
        return max(int(base_beams), 8)
    if max_len <= 24:
        return max(int(base_beams), 6)
    if max_len <= 40:
        return max(4, int(base_beams) - 1)
    return max(3, int(base_beams) - 2)


def retrieval_hits(clean_source: str, bundle: Dict[str, object] = None) -> List[Dict[str, object]]:
    bundle = memory_bundle if bundle is None else bundle
    memory_df_local = bundle["memory_df"]
    exact_memory_local = bundle["exact_memory"]
    vectorizer_local = bundle["vectorizer"]
    memory_matrix_local = bundle["memory_matrix"]
    source_token_sets_local = bundle["source_token_sets"]
    score_array_local = bundle["score_array"]
    score_max_local = float(bundle["score_max"])
    clean_source = normalize_ws(clean_source)
    if not clean_source:
        return []
    exact = exact_memory_local.get(clean_source, [])
    if exact:
        return [
            {"target": str(tgt), "score": 1.0, "similarity": 1.0, "memory_score": 1.0, "token_overlap": 1.0, "origin": "exact"}
            for tgt in exact[: max(1, config.retrieval_keep)]
        ]
    if (not config.use_translation_memory) or vectorizer_local is None or memory_matrix_local is None or len(memory_df_local) == 0:
        return []
    q = vectorizer_local.transform([clean_source])
    similarity_scores = cosine_similarity(q, memory_matrix_local).ravel()
    shortlist = similarity_scores.argsort()[::-1][: max(int(config.retrieval_shortlist), int(config.retrieval_topk), int(config.retrieval_keep))]
    query_tokens = source_token_keyset(clean_source)
    hits = []
    for idx in shortlist:
        idx = int(idx)
        sim = float(similarity_scores[idx])
        if sim < float(config.retrieval_min_score):
            continue
        row_score = float(score_array_local[idx]) / max(score_max_local, 1e-6)
        overlap = source_token_overlap(query_tokens, source_token_sets_local[idx])
        combined = sim + float(config.retrieval_domain_weight) * row_score + float(config.retrieval_token_overlap_weight) * overlap
        if combined < float(config.retrieval_combined_min_score):
            continue
        row = memory_df_local.iloc[idx]
        hits.append({
            "target": str(row["target"]),
            "score": round(combined, 6),
            "similarity": round(sim, 6),
            "memory_score": round(row_score, 6),
            "token_overlap": round(overlap, 6),
            "origin": str(row.get("origin", "")),
        })
    hits = sorted(
        hits,
        key=lambda hit: (-float(hit["score"]), -float(hit["similarity"]), -float(hit["memory_score"]), -float(hit["token_overlap"])),
    )
    out = []
    seen = set()
    for hit in hits:
        target = normalize_ws(hit["target"])
        if not target or target in seen:
            continue
        seen.add(target)
        out.append(hit)
        if len(out) >= int(config.retrieval_keep):
            break
    return out


def retrieval_candidates(clean_source: str, bundle: Dict[str, object] = None) -> List[str]:
    return [hit["target"] for hit in retrieval_hits(clean_source, bundle=bundle)]


def resolve_model_weights(model_weights: Dict[str, float] = None) -> Dict[str, float]:
    if model_weights is not None:
        return {str(k): float(v) for k, v in model_weights.items()}
    return {spec.label: float(getattr(spec, "weight", 1.0)) for spec in config.model_specs}


def corpus_bleu(predictions: List[str], references: List[str], max_n: int = 4) -> float:
    total_hyp_len = 0
    total_ref_len = 0
    clipped = [0.0] * max_n
    totals = [0.0] * max_n
    for hyp, ref in zip(predictions, references):
        hyp_tok = normalize_ws(hyp).split()
        ref_tok = normalize_ws(ref).split()
        total_hyp_len += len(hyp_tok)
        total_ref_len += len(ref_tok)
        for n in range(1, max_n + 1):
            hyp_ng = get_ngrams(hyp_tok, n)
            ref_ng = get_ngrams(ref_tok, n)
            for ng, cnt in hyp_ng.items():
                clipped[n - 1] += min(cnt, ref_ng.get(ng, 0))
                totals[n - 1] += cnt
    if total_hyp_len == 0:
        return 0.0
    bp = min(1.0, math.exp(1.0 - (total_ref_len / max(total_hyp_len, 1)))) if total_ref_len > 0 else 1.0
    log_avg = 0.0
    for n in range(max_n):
        if n == 0:
            if totals[n] == 0:
                return 0.0
            precision = clipped[n] / totals[n]
            if precision == 0:
                return 0.0
        else:
            precision = (clipped[n] + 1.0) / (totals[n] + 1.0)
        log_avg += math.log(max(precision, 1e-100)) / max_n
    return 100.0 * bp * math.exp(log_avg)


def corpus_chrfpp(predictions: List[str], references: List[str], max_char_n: int = 6, max_word_n: int = 2, beta: float = 2.0) -> float:
    total_prec_num = 0.0
    total_prec_den = 0.0
    total_rec_num = 0.0
    total_rec_den = 0.0
    for hyp, ref in zip(predictions, references):
        hyp = normalize_ws(hyp)
        ref = normalize_ws(ref)
        for n in range(1, max_char_n + 1):
            hyp_ng = char_ngrams(hyp, n)
            ref_ng = char_ngrams(ref, n)
            matched = sum(min(hyp_ng.get(ng, 0), ref_ng.get(ng, 0)) for ng in hyp_ng)
            total_prec_num += matched
            total_prec_den += sum(hyp_ng.values())
            total_rec_num += matched
            total_rec_den += sum(ref_ng.values())
        hyp_tok = hyp.split()
        ref_tok = ref.split()
        for n in range(1, max_word_n + 1):
            hyp_ng = get_ngrams(hyp_tok, n)
            ref_ng = get_ngrams(ref_tok, n)
            matched = sum(min(hyp_ng.get(ng, 0), ref_ng.get(ng, 0)) for ng in hyp_ng)
            total_prec_num += matched
            total_prec_den += sum(hyp_ng.values())
            total_rec_num += matched
            total_rec_den += sum(ref_ng.values())
    precision = total_prec_num / max(total_prec_den, 1e-100)
    recall = total_rec_num / max(total_rec_den, 1e-100)
    if precision + recall == 0:
        return 0.0
    beta2 = beta * beta
    return 100.0 * (1 + beta2) * precision * recall / (beta2 * precision + recall)


def evaluate_predictions(predictions: List[str], references: List[str]) -> Dict[str, float]:
    bleu = corpus_bleu(predictions, references)
    chrfpp = corpus_chrfpp(predictions, references)
    geo = math.sqrt(max(bleu, 0.0) * max(chrfpp, 0.0))
    return {
        "bleu": bleu,
        "chrfpp": chrfpp,
        "geo_mean_bleu_chrfpp": geo,
    }


def build_tuning_split():
    if not config.run_dev_tuning:
        return None, None
    doc_ids = [str(x) for x in gold_memory["oare_id"].dropna().astype(str).unique().tolist() if str(x).strip()]
    if len(doc_ids) < 4:
        return None, None
    rng = np.random.RandomState(config.tuning_seed)
    rng.shuffle(doc_ids)
    n_val = max(1, int(len(doc_ids) * float(config.tuning_val_ratio)))
    n_val = min(n_val, int(config.tuning_max_docs)) if int(config.tuning_max_docs) > 0 else n_val
    holdout_ids = set(doc_ids[:n_val])
    tuning_val_df = gold_memory[gold_memory["oare_id"].astype(str).isin(holdout_ids)].copy().reset_index(drop=True)
    tuning_train_gold = gold_memory[~gold_memory["oare_id"].astype(str).isin(holdout_ids)].copy()
    tuning_frames = [tuning_train_gold]
    if len(recon_memory):
        tuning_frames.append(recon_memory[~recon_memory["oare_id"].astype(str).isin(holdout_ids)].copy())
    if len(page_memory):
        tuning_frames.append(page_memory[~page_memory["oare_id"].astype(str).isin(holdout_ids)].copy())
    tuning_memory_df = pd.concat(tuning_frames, ignore_index=True).drop_duplicates(["source", "target"]).reset_index(drop=True)
    return tuning_val_df, build_memory_bundle(tuning_memory_df)


def candidate_weight_settings(model_specs: List[ModelSpec]):
    seen = set()
    settings = []
    labels = [spec.label for spec in model_specs]
    def add(name: str, weights: Dict[str, float]):
        weights = {label: float(weights.get(label, 1.0)) for label in labels}
        key = tuple((label, round(weights[label], 6)) for label in labels)
        if key not in seen:
            seen.add(key)
            settings.append((name, weights))
    add("current", {spec.label: float(getattr(spec, "weight", 1.0)) for spec in model_specs})
    if len(labels) > 1:
        add("uniform", {label: 1.0 for label in labels})
        for label in labels:
            add(f"boost_{label}", {name: (2.0 if name == label else 1.0) for name in labels})
            add(f"mild_{label}", {name: (1.5 if name == label else 0.75) for name in labels})
    return settings


memory_bundle = build_memory_bundle(memory_df)


class ModelWrapper:
    def __init__(self, spec: ModelSpec, runtime_config: EnsembleConfig):
        self.spec = spec
        self.config = runtime_config
        self.tokenizer = AutoTokenizer.from_pretrained(spec.model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(spec.model_path)
        self.model = self.model.to(self.config.device)
        self.model.eval()
        if self.config.use_better_transformer and self.config.device.type == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
            except Exception:
                pass
        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"Loaded {self.spec.label}: {num_params:,} parameters")

    def _build_inputs(self, clean_sources: List[str]) -> List[str]:
        prompts = []
        for src in clean_sources:
            prefix = self.spec.prompt_prefix
            if self.spec.use_domain_tags:
                tag = infer_domain_tag(src)
                prefix = (tag + " " if tag else "") + prefix
            prompts.append(prefix + src)
        return prompts

    def generate_candidates(self, clean_sources: List[str]) -> List[List[str]]:
        prompts = self._build_inputs(clean_sources)
        encoded = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.config.max_source_length,
        )
        encoded = {k: v.to(self.config.device) for k, v in encoded.items()}
        beam_size = adaptive_beam_size(clean_sources, self.spec.num_beams)
        beam_size = max(beam_size, int(self.spec.num_beam_candidates))
        pools = [[] for _ in clean_sources]
        with autocast_context():
            beam_out = self.model.generate(
                **encoded,
                do_sample=False,
                num_beams=int(beam_size),
                num_return_sequences=int(self.spec.num_beam_candidates),
                max_new_tokens=int(self.spec.max_new_tokens),
                repetition_penalty=float(self.spec.repetition_penalty),
                length_penalty=1.15,
                early_stopping=True,
                use_cache=True,
            )
            beam_txt = self.tokenizer.batch_decode(beam_out, skip_special_tokens=True)
        rb = int(self.spec.num_beam_candidates)
        for i in range(len(clean_sources)):
            pools[i].extend(beam_txt[i * rb:(i + 1) * rb])
        if int(self.spec.num_sample_candidates) > 0:
            with autocast_context():
                samp_out = self.model.generate(
                    **encoded,
                    do_sample=True,
                    num_beams=1,
                    top_p=float(self.spec.top_p),
                    temperature=float(self.spec.temperature),
                    num_return_sequences=int(self.spec.num_sample_candidates),
                    max_new_tokens=int(self.spec.max_new_tokens),
                    repetition_penalty=float(self.spec.repetition_penalty),
                    use_cache=True,
                )
                samp_txt = self.tokenizer.batch_decode(samp_out, skip_special_tokens=True)
            rs = int(self.spec.num_sample_candidates)
            for i in range(len(clean_sources)):
                pools[i].extend(samp_txt[i * rs:(i + 1) * rs])
        return [dedup_keep_order(postprocessor.postprocess_batch(pool)) for pool in pools]


models = [ModelWrapper(spec, config) for spec in config.model_specs]


def translate_sources(clean_sources: List[str], *, active_models=None, memory_bundle_override=None, model_weights: Dict[str, float] = None, mbr_pool_cap: int = None, save_debug: bool = None, show_progress: bool = True):
    active_models = models if active_models is None else active_models
    memory_bundle_local = memory_bundle if memory_bundle_override is None else memory_bundle_override
    resolved_weights = resolve_model_weights(model_weights)
    if mbr_pool_cap is None:
        mbr_pool_cap = config.mbr_pool_cap
    if save_debug is None:
        save_debug = config.save_candidate_debug
    predictions = []
    debug_rows = []
    iterator = range(0, len(clean_sources), config.batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Translating")
    for start in iterator:
        batch_sources = clean_sources[start:start + config.batch_size]
        pooled = [[] for _ in batch_sources]
        pooled_prior = [{} for _ in batch_sources]
        for wrapper in active_models:
            wrapper_weight = float(resolved_weights.get(wrapper.spec.label, getattr(wrapper.spec, "weight", 1.0)))
            wrapper_pools = wrapper.generate_candidates(batch_sources)
            for i, cand_list in enumerate(wrapper_pools):
                for cand in cand_list:
                    pooled[i].append(cand)
                    pooled_prior[i][cand] = pooled_prior[i].get(cand, 0.0) + wrapper_weight
        for i, src in enumerate(batch_sources):
            memory_hits = retrieval_hits(src, bundle=memory_bundle_local)
            raw_memory_targets = [hit["target"] for hit in memory_hits]
            processed_memory_targets = postprocessor.postprocess_batch(raw_memory_targets) if raw_memory_targets else []
            memory_prior = {}
            for hit, cand in zip(memory_hits, processed_memory_targets):
                cand = normalize_ws(cand)
                if not cand:
                    continue
                memory_prior[cand] = max(memory_prior.get(cand, 0.0), float(config.memory_candidate_weight) * float(hit["score"]))
            memory_cands = dedup_keep_order(processed_memory_targets)
            for cand in memory_cands:
                pooled[i].append(cand)
                pooled_prior[i][cand] = pooled_prior[i].get(cand, 0.0) + float(memory_prior.get(cand, config.memory_candidate_weight))
            pooled[i] = sort_candidates_by_prior(postprocessor.postprocess_batch(pooled[i]), pooled_prior[i])
            exact = memory_bundle_local["exact_memory"].get(src, [])
            if exact:
                chosen = postprocessor.postprocess_batch([exact[0]])[0]
            elif config.use_mbr:
                chosen = mbr_pick(pooled[i], prior_scores=pooled_prior[i], pool_cap=mbr_pool_cap)
            else:
                chosen = pooled[i][0] if pooled[i] else "translation unavailable"
            predictions.append(chosen)
            if save_debug:
                debug_rows.append({
                    "row_index": start + i,
                    "source": src,
                    "chosen": chosen,
                    "candidate_count": len(pooled[i]),
                    "candidate_priors": json.dumps({cand: float(pooled_prior[i].get(cand, 0.0)) for cand in pooled[i][:12]}, ensure_ascii=False),
                    "memory_hits": json.dumps(memory_hits, ensure_ascii=False),
                    "memory_candidates": json.dumps(memory_cands, ensure_ascii=False),
                    "all_candidates": json.dumps(pooled[i], ensure_ascii=False),
                })
    return predictions, pd.DataFrame(debug_rows)


def tune_ensemble_settings(active_models=None):
    active_models = models if active_models is None else active_models
    tuning_val_df, tuning_bundle = build_tuning_split()
    if tuning_val_df is None or len(tuning_val_df) == 0:
        print("Dev tuning skipped: not enough held-out gold rows.")
        return None, pd.DataFrame()
    weight_settings = candidate_weight_settings([wrapper.spec for wrapper in active_models])
    pool_caps = []
    for cap in [config.mbr_pool_cap] + list(config.tuning_mbr_pool_caps):
        if cap is None:
            continue
        cap = int(cap)
        if cap not in pool_caps:
            pool_caps.append(cap)
    refs = tuning_val_df["target"].astype(str).tolist()
    rows = []
    for weight_name, weight_map in weight_settings:
        for pool_cap in pool_caps:
            preds, _ = translate_sources(
                tuning_val_df["source"].astype(str).tolist(),
                active_models=active_models,
                memory_bundle_override=tuning_bundle,
                model_weights=weight_map,
                mbr_pool_cap=pool_cap,
                save_debug=False,
                show_progress=False,
            )
            metrics = evaluate_predictions(preds, refs)
            rows.append({
                "setting": weight_name,
                "mbr_pool_cap": int(pool_cap),
                "model_weights": json.dumps(weight_map, ensure_ascii=False, sort_keys=True),
                "bleu": metrics["bleu"],
                "chrfpp": metrics["chrfpp"],
                "geo_mean_bleu_chrfpp": metrics["geo_mean_bleu_chrfpp"],
                "val_rows": int(len(tuning_val_df)),
            })
    tuning_results = pd.DataFrame(rows).sort_values("geo_mean_bleu_chrfpp", ascending=False).reset_index(drop=True)
    if len(tuning_results) == 0:
        return None, tuning_results
    best = {k: (v.item() if hasattr(v, "item") else v) for k, v in tuning_results.iloc[0].to_dict().items()}
    best_weights = json.loads(best["model_weights"])
    for spec in config.model_specs:
        spec.weight = float(best_weights.get(spec.label, getattr(spec, "weight", 1.0)))
    config.mbr_pool_cap = int(best["mbr_pool_cap"])
    best["model_weights"] = best_weights
    print("Best tuning setting:", best)
    return best, tuning_results


In [ ]:
best_tuning = None
tuning_results = pd.DataFrame()
if config.run_dev_tuning:
    best_tuning, tuning_results = tune_ensemble_settings()
    if len(tuning_results):
        tuning_results.to_csv(config.tuned_debug_csv, index=False)
        print("Saved tuning results:", config.tuned_debug_csv)

preds, candidate_debug = translate_sources(test_df["source"].tolist())

submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": preds,
})
submission.to_csv(config.submission_path, index=False)

if config.save_candidate_debug:
    candidate_debug.to_csv(config.debug_candidate_csv, index=False)

config_payload = asdict(config)
config_payload["device"] = str(config.device)
config_payload["model_specs"] = [asdict(spec) for spec in config.model_specs]
config_payload["best_tuning"] = best_tuning
with open(config.config_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2, ensure_ascii=False)

print("Saved submission:", config.submission_path)
print("Saved config:", config.config_path)
if config.save_candidate_debug:
    print("Saved candidates:", config.debug_candidate_csv)
submission.head()
